# Tutorial on quantEM `Vector` class

This tutorial demonstrates how the quantEM `Vector` module works 

Colin Ophus and Arthur McCray
March 5, 2026

In [22]:
import numpy as np
import torch

import quantem as em
from quantem.core.datastructures import Vector


## Creating a Vector

A `Vector` stores ragged per-cell data on a fixed grid. Each cell holds a variable number of rows, one value per named field.

`Vector` is torch-backed: rows exposed by `.tensor` and `.flatten()` are torch tensors, and arithmetic returns new torch-backed `Vector` objects. NumPy arrays, Python lists and scalars are still accepted as *input* anywhere a payload is taken — they are converted at the boundary.

Create an empty `Vector` with `from_shape`, then assign cell data with `[]`:

In [2]:
Nx, Ny = 20, 30  # fixed-grid dimensions (e.g. scan positions)

v = Vector.from_shape(
    shape=(Nx, Ny),
    fields=("kx", "ky", "intensity"),
    units=("A^-1", "A^-1", "counts"),
    name="diffraction_vectors",
)

# Assign each cell a 2D array of shape (n_rows, num_fields).
# These are NumPy arrays; they are converted to tensors on assignment.
rng = np.random.default_rng(42)
for rx in range(Nx):
    for ry in range(Ny):
        n = rng.integers(5, 20)
        phi = rng.random(n) * 2 * np.pi
        r = 10 + rng.standard_normal(n) * 2
        v[rx, ry] = np.column_stack((r * np.cos(phi), r * np.sin(phi), rng.random(n)))

print(v)

quantem.Vector, shape=(20, 30), name=diffraction_vectors
  fields = ['kx', 'ky', 'intensity']
  units: ['A^-1', 'A^-1', 'counts']
  dtype: torch.float32, device: cpu


When many cells are already available, collecting them first and calling `from_data` is more efficient. Repeated `v[rx, ry] = ...` assignments are convenient for incremental results, but each assignment grows the packed row buffer. `from_data` normalizes all cells and concatenates their rows in one bulk operation:

In [3]:
def make_diffraction_cell(rng):
    n = rng.integers(5, 20)
    phi = rng.random(n) * 2 * np.pi
    r = 10 + rng.standard_normal(n) * 2
    return np.column_stack((r * np.cos(phi), r * np.sin(phi), rng.random(n)))


bulk_rng = np.random.default_rng(42)
cells = [
    [make_diffraction_cell(bulk_rng) for _ in range(Ny)]
    for _ in range(Nx)
]
v_bulk = Vector.from_data(
    cells,
    fields=("kx", "ky", "intensity"),
    units=("A^-1", "A^-1", "counts"),
    name="diffraction_vectors_bulk",
)
print(v_bulk)

quantem.Vector, shape=(20, 30), name=diffraction_vectors_bulk
  fields = ['kx', 'ky', 'intensity']
  units: ['A^-1', 'A^-1', 'counts']
  dtype: torch.float32, device: cpu


Alternatively, create from existing nested data with `from_data`. Cells can be lists, tuples, NumPy arrays, or torch tensors:

In [4]:
v_small = Vector.from_data(
    data=[
        np.array([[1.0, 2.0], [3.0, 4.0]]),   # cell 0: 2 rows (NumPy)
        torch.tensor([[5.0, 6.0], [7.0, 8.0], [9.0, 10.0]]),  # cell 1: 3 rows (torch)
    ],
    fields=["x", "y"],
    units=["m", "m"],
    name="example",
)
print(v_small)
print("\ncell 0:\n", v_small[0].tensor)
print("\ncell 1:\n", v_small[1].tensor)

quantem.Vector, shape=(2,), name=example
  fields = ['x', 'y']
  units: ['m', 'm']
  dtype: torch.float32, device: cpu

cell 0:
 tensor([[1., 2.],
        [3., 4.]])

cell 1:
 tensor([[ 5.,  6.],
        [ 7.,  8.],
        [ 9., 10.]])


## Properties

In [5]:
print("shape:      ", v.shape)
print("fields:     ", v.fields)
print("units:      ", v.units)
print("num_fields: ", v.num_fields)
print("num_cells:  ", v.num_cells)
print("total_rows: ", v.total_rows)
print("dtype:      ", v.dtype)
print("device:     ", v.device)
print("name:       ", v.name)

shape:       (20, 30)
fields:      ['kx', 'ky', 'intensity']
units:       ['A^-1', 'A^-1', 'counts']
num_fields:  3
num_cells:   600
total_rows:  7243
dtype:       torch.float32
device:      cpu
name:        diffraction_vectors


## dtype and device

The row buffer defaults to `torch.float32` on the CPU. Pass `dtype=` / `device=` to change either, and use `.to(device)` to move an existing `Vector`.

It's worth pointing out that `.to()` moves storage that is **shared by every view** of the same `Vector`, so moving one view moves them all.

`.numpy()` returns a detached NumPy array copy on the CPU; it is equivalent to `.flatten().detach().cpu().numpy()`.

In [6]:
# Ask for double precision at construction
v_double = Vector.from_shape(shape=(2,), fields=["x", "y"], dtype=torch.float64)
v_double[0] = np.array([[1.0, 2.0]])
print("explicit dtype: ", v_double.dtype)

# Back to NumPy when you need it
print("\nnumpy() ->", type(v_small.numpy()).__name__, v_small.numpy().shape)

# Move to the GPU if one is available
if torch.cuda.is_available():
    v_gpu = v_small.copy().to("cuda:0")
    print("\nmoved to:", v_gpu.device)
    print("arithmetic stays on device:", (v_gpu.select_fields("x") * 2).flatten().device)
else:
    print("\nno CUDA available; .to('cuda') skipped")

explicit dtype:  torch.float64

numpy() -> ndarray (5, 2)

moved to: cuda:0
arithmetic stays on device: cuda:0


## Fixed-grid indexing

`[]` selects along the fixed-grid axes using NumPy-like indexing and always returns a `Vector` view over shared storage.

- Integer index → 0D `Vector`; access the cell tensor with `.tensor`
- Slice/fancy index → sub-grid `Vector` (indices can be lists, NumPy arrays, or torch tensors)
- `.flatten()` concatenates all selected cells row-wise into a 2D torch tensor

Repeated fancy indices are valid for reading, reordering, sampling, and out-of-place arithmetic. Write-through operations reject repeated cell indices because assigning or modifying the same backing cell more than once would be ambiguous.

In [7]:
# 0D selection → use .tensor to get the torch tensor for that cell
cell = v[0, 0]
print("cell shape:", cell.shape)       # () means scalar / 0D
print("cell tensor:\n", cell.tensor)

# Slice along one or both axes → returns a sub-grid Vector
row = v[0, :]   # first row, all columns
print("\nrow shape:", row.shape)

# Flatten all selected cells into one 2D tensor
flat = v[0, :].flatten()
print("flattened shape:", flat.shape)   # (total_rows, num_fields)

# Copy data from one region to another (write-through)
v[1, :] = v[0, :]

cell shape: ()
cell tensor:
 tensor([[ -8.6852,   3.5097,   0.8228],
        [  6.2849,  -7.7349,   0.4434],
        [ -2.6930,  -7.8445,   0.2272],
        [  9.7595,   6.5591,   0.5546],
        [ 11.4203,  -1.7630,   0.0638],
        [  0.7086, -10.1073,   0.8276]])

row shape: (30,)
flattened shape: torch.Size([384, 3])


## Field selection & arithmetic

`select_fields(...)` returns a **write-through view** over a subset of fields. Changes made through the view are reflected in the parent `Vector`.

Arithmetic operators (`+`, `-`, `*`, `/`, `**`, `%`, `//`, unary `-`, `abs`) all work on `Vector` objects and return new `Vector` instances. In-place operators (`+=`, `*=`, etc.) modify the backing data directly.

In [8]:
v2 = v.copy() 
kx = v2.select_fields("kx")
ky = v2.select_fields("ky")
intensity = v2.select_fields("intensity")

# In-place: modifies v directly
kx += 16
ky += 16

# Arithmetic between field views
r_squared = kx**2 + ky**2        # new Vector, field value is "kx" same as kx vector
r_squared.rename_fields({"kx": "r_squared"})
r_squared.name = "r_squared"
print(r_squared)
print("r² first 5 values:", r_squared.flatten()[:5, 0])

# Scale intensity in-place by a per-row factor
scale = kx.flatten() / r_squared.flatten()
intensity.set_flattened(intensity.flatten() * scale)

# Assign using [...]
kx[...] = torch.abs(kx.flatten())   # equivalent to abs(kx)[...] = ...

print("\nv fields after modifications:", v2.fields)
print("kx range: [{:.2f}, {:.2f}]".format(kx.flatten().min().item(), kx.flatten().max().item()))

quantem.Vector, shape=(20, 30), name=r_squared
  fields = ['r_squared']
  units: ['A^-1']
  dtype: torch.float32, device: cpu
r² first 5 values: tensor([ 434.1351,  564.9296,  243.5869, 1172.4636,  954.5634])

v fields after modifications: ['kx', 'ky', 'intensity']
kx range: [0.40, 31.14]


## torch function support

Torch functions work directly on `Vector` objects via `__torch_function__`. Each direct `Vector` argument is replaced by its flattened rows before the function is applied. Results from explicitly supported elementwise functions are rebuilt into a `Vector`; other functions return ordinary tensors so that shape-preserving operations cannot silently move rows between ragged cells.

- Elementwise functions (`torch.sin`, `torch.maximum`, ...) return a `Vector`
- Multi-output functions (e.g. `torch.frexp`) return a tuple of `Vector`s
- **Reductions** (`torch.sum`, `torch.mean`, ...) and structural operations (`torch.flip`, `torch.roll`, ...) pass straight through as plain tensors

NumPy ufuncs are deliberately disabled: `np.sin(vector)` raises a `TypeError` pointing you at the torch equivalent.

In [9]:
kx = v.select_fields("kx")
ky = v.select_fields("ky")
intensity = v.select_fields("intensity")
print("kx[:2]:\n", kx.flatten()[:2])

print("\nsin(kx)[:2]:\n", torch.sin(kx).flatten()[:2])
print("\nmaximum(kx, 10)[:2]:\n", torch.maximum(kx, torch.tensor(10.0)).flatten()[:2])

# Multi-output: frexp returns (mantissa, exponent) as a tuple of Vectors
mantissa, exponent = torch.frexp(kx)
print("\nfrexp mantissa[:2]:", mantissa.flatten()[:2, 0])
print("frexp exponent[:2]:", exponent.flatten()[:2, 0])

# Reductions return a plain tensor rather than a Vector
print("\nmean intensity:", torch.mean(intensity), type(torch.mean(intensity)).__name__)

# NumPy ufuncs are turned off on purpose
try:
    np.sin(kx)
except TypeError as err:
    print("\nnp.sin(kx) ->", type(err).__name__)

kx[:2]:
 tensor([[-8.6852],
        [ 6.2849]])

sin(kx)[:2]:
 tensor([[-0.6740],
        [ 0.0017]])

maximum(kx, 10)[:2]:
 tensor([[10.],
        [10.]])

frexp mantissa[:2]: tensor([-0.5428,  0.7856])
frexp exponent[:2]: tensor([4, 3], dtype=torch.int32)

mean intensity: tensor(0.5008) Tensor

np.sin(kx) -> TypeError


## Row-wise updates with `set_flattened`

`set_flattened(values)` writes back into all selected cells without changing per-cell row counts. This is the natural companion to `flatten()` for applying row-wise torch transforms.

In [10]:
v2 = v.copy()
kx = v2.select_fields("kx")
ky = v2.select_fields("ky")

# Zero-out kx for any row within radius 10 of the origin
mask = (kx.flatten()**2 + ky.flatten()**2) < 100  # shape (total_rows, 1)
kx.set_flattened(torch.where(mask, 0.0, kx.flatten()))

print(f"rows zeroed: {int(mask.sum())} / {v2.total_rows}")

rows zeroed: 3599 / 7229


## Schema operations

`add_fields` and `remove_fields` modify the field schema for the whole `Vector`. `append_rows` adds rows to a single cell.

In [11]:
kx = v.select_fields("kx")
ky = v.select_fields("ky")

# Add a derived field with initial values
v.add_fields("r", values=torch.sqrt(kx**2 + ky**2), units="A^-1")
print("fields after add:", v.fields)

# Remove it again
v.remove_fields("r")
print("fields after remove:", v.fields)

# Append rows to a single cell
before = v[0, 0].tensor.shape[0]
v.append_rows((0, 0), np.array([[1.0, 2.0, 0.5]]))
after = v[0, 0].tensor.shape[0]
print(f"\ncell [0,0] rows: {before} → {after}")

fields after add: ['kx', 'ky', 'intensity', 'r']
fields after remove: ['kx', 'ky', 'intensity']

cell [0,0] rows: 6 → 7


## Exporting to a DataFrame

`to_polars()` flattens a `Vector` into a [polars](https://pola.rs) DataFrame: one row per ragged row, with the fixed-grid location carried in leading integer columns (`dim_0`, `dim_1`, ...) followed by one column per field.

This is the easiest way to inspect, filter, group, or join ragged data, since the fixed-grid location travels with every row instead of being implicit in the nesting.

polars is an optional dependency — install it with `pip install polars` (or `pip install "quantem[dataframe]"`).

In [12]:
import polars as pl

df = v.to_polars()
print("shape:", df.shape)  # (total_rows, grid_ndim + num_fields)
df.head(8)

shape: (7230, 5)


dim_0,dim_1,kx,ky,intensity
i64,i64,f32,f32,f32
0,0,-8.685178,3.509703,0.822762
0,0,6.284925,-7.734908,0.443414
0,0,-2.693048,-7.844519,0.227239
0,0,9.759505,6.559066,0.554585
0,0,11.420297,-1.763048,0.063817
0,0,0.708593,-10.107253,0.827631
0,0,1.0,2.0,0.5
0,1,-8.297236,-9.022919,0.664851


The export honors the current selection, so both `[]` and `select_fields(...)` narrow the table. Grid coordinates are always reported in the **root** grid rather than the view's local coordinates, so a row always tells you which scan position it actually came from:

In [13]:
# A single cell, with only two of the three fields
print(v[3, 5].select_fields("kx", "ky").to_polars())

# Rename the fixed-grid columns to something meaningful for this dataset
print(v[3, 5:7].to_polars(dim_names=("scan_x", "scan_y")))

shape: (6, 4)
┌───────┬───────┬────────────┬────────────┐
│ dim_0 ┆ dim_1 ┆ kx         ┆ ky         │
│ ---   ┆ ---   ┆ ---        ┆ ---        │
│ i64   ┆ i64   ┆ f32        ┆ f32        │
╞═══════╪═══════╪════════════╪════════════╡
│ 3     ┆ 5     ┆ 0.966149   ┆ -12.845343 │
│ 3     ┆ 5     ┆ 6.299318   ┆ 7.65857    │
│ 3     ┆ 5     ┆ -4.212517  ┆ 11.243736  │
│ 3     ┆ 5     ┆ -11.458661 ┆ 4.065883   │
│ 3     ┆ 5     ┆ -4.186042  ┆ 8.569332   │
│ 3     ┆ 5     ┆ -8.486384  ┆ 7.292703   │
└───────┴───────┴────────────┴────────────┘
shape: (22, 5)
┌────────┬────────┬────────────┬────────────┬───────────┐
│ scan_x ┆ scan_y ┆ kx         ┆ ky         ┆ intensity │
│ ---    ┆ ---    ┆ ---        ┆ ---        ┆ ---       │
│ i64    ┆ i64    ┆ f32        ┆ f32        ┆ f32       │
╞════════╪════════╪════════════╪════════════╪═══════════╡
│ 3      ┆ 5      ┆ 0.966149   ┆ -12.845343 ┆ 0.401699  │
│ 3      ┆ 5      ┆ 6.299318   ┆ 7.65857    ┆ 0.722478  │
│ 3      ┆ 5      ┆ -4.212517  ┆ 11.2

Because the grid location is now an ordinary column, the full polars API is available. Grouping on the `dim_*` columns gives per-scan-position statistics, which would otherwise require looping over cells:

In [14]:
# Per-scan-position summary: how many peaks, and how bright are they?
per_position = (
    df.group_by("dim_0", "dim_1")
    .agg(
        pl.len().alias("n_peaks"),
        pl.col("intensity").sum().alias("total_intensity"),
        (pl.col("kx") ** 2 + pl.col("ky") ** 2).sqrt().mean().alias("mean_radius"),
    )
    .sort("dim_0", "dim_1")
)
print(per_position.head())

# Row counts agree with the Vector's own bookkeeping
print("\nmatches row_counts():", per_position["n_peaks"].to_list() == v.row_counts())

# Filtering works on fields and grid location together
bright_peak = df.filter(
    (pl.col("intensity") > 0.9) & (pl.col("dim_0") < 5)
)
print(f"\n{bright_peak.height} bright peaks in the first 5 scan rows")

shape: (5, 5)
┌───────┬───────┬─────────┬─────────────────┬─────────────┐
│ dim_0 ┆ dim_1 ┆ n_peaks ┆ total_intensity ┆ mean_radius │
│ ---   ┆ ---   ┆ ---     ┆ ---             ┆ ---         │
│ i64   ┆ i64   ┆ u32     ┆ f32             ┆ f32         │
╞═══════╪═══════╪═════════╪═════════════════╪═════════════╡
│ 0     ┆ 0     ┆ 7       ┆ 3.439448        ┆ 9.044333    │
│ 0     ┆ 1     ┆ 16      ┆ 7.984736        ┆ 10.415007   │
│ 0     ┆ 2     ┆ 10      ┆ 4.136971        ┆ 9.640105    │
│ 0     ┆ 3     ┆ 11      ┆ 3.878064        ┆ 9.64804     │
│ 0     ┆ 4     ┆ 11      ┆ 5.483243        ┆ 9.300777    │
└───────┴───────┴─────────┴─────────────────┴─────────────┘

matches row_counts(): True

186 bright peaks in the first 5 scan rows


## File I/O

`Vector` can be saved and loaded using the standard quantEM I/O interface.

The row buffer is written to disk as a compressed Zarr array rather than a torch blob, and converted back to a tensor on load. Two consequences worth knowing: a `Vector` saved on the GPU always loads back onto the **CPU** (call `.to(...)` to move it again), and files written before the torch migration still load, keeping their original dtype.

In [15]:
import tempfile

with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp:
    path = tmp.name

try:
    v.save(path, mode="o")
    v_loaded = em.io.load(path)

    # Verify round-trip
    print("loaded:", v_loaded)
    torch.testing.assert_close(v[3, 5].tensor, v_loaded[3, 5].tensor)
    print("Round-trip OK")
finally:
    import os
    if os.path.exists(path):
        os.remove(path)

loaded: quantem.Vector, shape=(20, 30), name=diffraction_vectors
  fields = ['kx', 'ky', 'intensity']
  units: ['A^-1', 'A^-1', 'counts']
  dtype: torch.float32, device: cpu
Round-trip OK


-- end notebook --